# Engagement Analysis Notebook

This notebook provides a starting point for academic researchers to analyze data collected by the `EduIG-Pipeline`.

## Prerequisites
Ensure you have installed the research dependencies:
```bash
pip install -r requirements-research.txt
```

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting styles
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data from SQLite
Connect to the local SQLite database and load the `profiles` and `posts` tables into pandas DataFrames.

In [ ]:
DB_PATH = '../data/eduig.db'

with sqlite3.connect(DB_PATH) as conn:
    profiles_df = pd.read_sql("SELECT * FROM profiles", conn)
    posts_df = pd.read_sql("SELECT * FROM posts", conn)

print(f"Loaded {len(profiles_df)} profiles.")
print(f"Loaded {len(posts_df)} posts.")

## 2. Engagement Rate Distribution
Analyze the distribution of engagement rates across all extracted posts.

In [ ]:
# Drop posts with missing engagement rates
valid_posts = posts_df.dropna(subset=['engagement_rate'])

sns.histplot(valid_posts['engagement_rate'], bins=50, kde=True, color='blue')
plt.title('Distribution of Post Engagement Rates')
plt.xlabel('Engagement Rate (Likes + Comments / Followers)')
plt.ylabel('Frequency')
plt.show()

## 3. Followers vs. Average Engagement Rate
Does having more followers correlate with a lower average engagement rate?

In [ ]:
# Calculate average engagement rate per profile
avg_engagement = valid_posts.groupby('profile_id')['engagement_rate'].mean().reset_index()
avg_engagement.rename(columns={'engagement_rate': 'avg_engagement_rate'}, inplace=True)

# Merge with profile data to get follower counts
merged_df = pd.merge(avg_engagement, profiles_df, on='profile_id')

sns.scatterplot(data=merged_df, x='followers', y='avg_engagement_rate', alpha=0.7)
plt.xscale('log')  # Log scale for followers as distribution is heavily skewed
plt.title('Followers vs. Average Engagement Rate')
plt.xlabel('Followers (Log Scale)')
plt.ylabel('Average Engagement Rate')
plt.show()